In [ ]:
import os
import wave
import numpy as np
import torch
import torchaudio
from pathlib import Path
from transformers import AutoProcessor, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score


In [ ]:
CREMA_DIR = Path(r"C:\Users\acer\Downloads\AudioWAV")

MAX_CLIPS = None

EMOTION_MAP = {'ANG': 0, 'DIS': 1, 'FEA': 2, 'HAP': 3, 'NEU': 4, 'SAD': 5}
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


In [ ]:
print("Loading Ultravox model (this may take a minute)...")
model = AutoModel.from_pretrained(
    "fixie-ai/ultravox-v0_5-llama-3_2-1b",
    trust_remote_code=True,
    dtype="auto",
    token=True
)
processor = AutoProcessor.from_pretrained(
    "fixie-ai/ultravox-v0_5-llama-3_2-1b",
    trust_remote_code=True
)

model.eval()
model.to(device)
print("Model loaded successfully!")


In [ ]:

representations = {}

def make_hook(layer_name):
    def hook_fn(module, input, output):
        if isinstance(output, tuple):
            tensor_output = output[0]
        else:
            tensor_output = output
        
        representations[layer_name] = tensor_output.detach().cpu().float()
    return hook_fn

p1_hook = model.audio_tower.layers[-1].register_forward_hook(make_hook('P1'))
p2_hook = model.multi_modal_projector.ln_post.register_forward_hook(make_hook('P2'))

print("Registered hooks for P1 (Audio Encoder) and P2 (Projector/Adapter).")


In [ ]:
def load_wav(path):
    with wave.open(str(path), 'rb') as wf:
        sr = wf.getframerate()
        nchannels = wf.getnchannels()
        sampwidth = wf.getsampwidth()
        nframes = wf.getnframes()
        raw = wf.readframes(nframes)

    if sampwidth == 1:
        audio = np.frombuffer(raw, dtype=np.uint8).astype(np.float32)
        audio = (audio - 128.0) / 128.0
    elif sampwidth == 2:
        audio = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0
    elif sampwidth == 3:
        raw_bytes = np.frombuffer(raw, dtype=np.uint8).reshape(-1, 3)
        audio = (raw_bytes[:, 0].astype(np.int32) |
                 (raw_bytes[:, 1].astype(np.int32) << 8) |
                 (raw_bytes[:, 2].astype(np.int32) << 16))
        audio = (audio.astype(np.int32) - 2**23) / float(2**23)
    elif sampwidth == 4:
        audio = np.frombuffer(raw, dtype=np.int32).astype(np.float32) / 2147483648.0
    else:
        raise ValueError(f"Unsupported sample width: {sampwidth}")

    if nchannels > 1:
        audio = audio.reshape(-1, nchannels).T
    else:
        audio = audio[np.newaxis, :]

    return audio, sr


In [ ]:
files = sorted(list(CREMA_DIR.glob("*.wav")))
if MAX_CLIPS is not None:
    files = files[:MAX_CLIPS]

print(f"Extracting representations from {len(files)} clips...")

X_p1 = []
X_p2 = []
y_emotion = []
y_speaker = []
y_sentence = []

for i, wav_file in enumerate(files):
    if (i + 1) % 100 == 0 or i == 0 or i == len(files) - 1:
        print(f"  Processed {i + 1}/{len(files)} clips...")

    parts = wav_file.stem.split('_')
    if len(parts) < 3:
        continue

    actor_id = parts[0]   # Speaker id
    sentence = parts[1]   # Semantic / lexical
    emotion = parts[2]    # Emotion

    if emotion not in EMOTION_MAP:
        continue

    try:
        waveform, sr = load_wav(wav_file)
    except Exception as e:
        print(f"  Skipping {wav_file.name}: Load error {e}")
        continue

    if sr != 16000:
        waveform_tensor = torch.from_numpy(waveform)
        waveform = torchaudio.transforms.Resample(sr, 16000)(waveform_tensor).numpy()
        sr = 16000

    # Convert to mono if stereo
    if waveform.shape[0] > 1:
        waveform = waveform.mean(axis=0, keepdims=True)

    waveform_np = waveform.squeeze().astype(np.float32)

    try:
        inputs = processor(
            text="<|audio|>",
            audio=waveform_np,
            sampling_rate=16000,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            model(**inputs)
            
    except Exception as e:
        print(f"  Skipping {wav_file.name}: Model error {e}")
        continue

    if 'P1' not in representations or 'P2' not in representations:
        print("Error: Hooks are not firing! Please check layer structure.")
        break

    p1_vec = representations['P1'].squeeze(0).mean(dim=0).numpy()
    p2_vec = representations['P2'].squeeze(0).mean(dim=0).numpy()

    X_p1.append(p1_vec)
    X_p2.append(p2_vec)
    y_emotion.append(EMOTION_MAP[emotion])
    y_speaker.append(int(actor_id))
    y_sentence.append(sentence)

p1_hook.remove()
p2_hook.remove()

X_p1 = np.array(X_p1)
X_p2 = np.array(X_p2)
y_emotion = np.array(y_emotion)
y_speaker = np.array(y_speaker)

sentence_encoder = LabelEncoder()
y_sentence = sentence_encoder.fit_transform(y_sentence)

print(f"Successfully extracted representations for {len(X_p1)} clips!")


In [ ]:
np.save('X_p1.npy', X_p1)
np.save('X_p2.npy', X_p2)
np.save('y_emotion.npy', y_emotion)
np.save('y_speaker.npy', y_speaker)
np.save('y_sentence.npy', y_sentence)
print("Saved feature arrays to disk.")


In [ ]:
def evaluate_probe(X, y, property_name, layer_name):
    #80-20 
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    clf.fit(X_train_scaled, y_train)

    y_pred = clf.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    
    print(f"[{property_name:8s}] Layer: {layer_name:12s} | Test Accuracy: {acc * 100:.2f}%")
    return acc


In [ ]:
print("\n" + "="*50)
print("RUNNING PROBING CLASSIFIERS (Simple Train/Test Split)")
print("="*50)

print("\n- 1. Probing for Emotion ")
evaluate_probe(X_p1, y_emotion, "Emotion", "P1 (Encoder)")
evaluate_probe(X_p2, y_emotion, "Emotion", "P2 (Projector)")

print("\n- 2. Probing for Speaker Identity ")
evaluate_probe(X_p1, y_speaker, "Speaker", "P1 (Encoder)")
evaluate_probe(X_p2, y_speaker, "Speaker", "P2 (Projector)")

print("\n- 3. Probing for Semantic Content")
evaluate_probe(X_p1, y_sentence, "Semantic", "P1 (Encoder)")
evaluate_probe(X_p2, y_sentence, "Semantic", "P2 (Projector)")

print("\n"+"="*50)
print("Probing experiment finished successfully!")
print("="*50)
